# Lab 3: Designing Hybrid Context Layers


## Goal

Combine the three context layers — vector search (semantic), knowledge graph (relational),
and metadata (structured) — into a unified hybrid query function. Then use that function
to answer a complex financial intelligence question that no single layer could answer alone.

## What you'll do

1. See why each context layer alone is insufficient for complex queries
3. Build a `hybrid_context_query()` function that combines all three layers
4. Answer a complex multi-company financial intelligence question

---

In [0]:
%run ../utils/config

In [0]:
KA_AGENT_NAME  = "financial-intelligence-assistant"   # Knowledge Assistant
VS_INDEX_NAME     = f"{catalog}.{shared_schema}.financial_docs_for_search_index"

## Part 1: Why each layer alone is insufficient

Let's demonstrate the limitations of each layer in isolation.

In [0]:
# The question we want to answer:
QUESTION = (
    "What are the common AI infrastructure investment themes across Apple, Amazon, "
    "Microsoft, NVIDIA, and Meta, and who are the key technology partners each company "
    "is most reliant on for their AI strategy?"
)

In [0]:
# =================================================
# Layer 1 only: Vector search — good at semantic similarity, misses explicit relationships
# =================================================
import pandas as pd
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

result_columns = ["company", "fiscal_period", "document_type", "chunk_text", "score"]

vector_results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text=QUESTION,
    columns=["company", "fiscal_period", "document_type", "chunk_text"],
    num_results=5,
)

print("Vector layer alone — top 5 retrieved chunks:")
display(pd.DataFrame(vector_results.result.data_array or [], columns=result_columns))

print("\n⚠️  Missing: explicit partnership names, cross-company comparison, relationship strength")

In [0]:
# =================================================
# Layer 2 only: Graph — precise relationships, but loses narrative context
# =================================================
graph_results = spark.sql(f"""
    SELECT subject AS company, predicate, object AS partner, COUNT(*) AS mention_count
    FROM relationships
    WHERE subject IN ('Apple', 'Amazon', 'Microsoft', 'NVIDIA', 'Meta')
      AND predicate IN ('partners_with', 'invests_in')
    GROUP BY subject, predicate, object
    ORDER BY subject, mention_count DESC
    LIMIT 20
""").collect()

print("Graph layer alone — top partnerships:")
for row in graph_results[:15]:
    print(f"  {(row['company'] or ''):<12} → {(row['predicate'] or ''):<15} → {(row['partner'] or ''):<50} ({(row['mention_count'] or '')} mentions)")

print("\n⚠️  Missing: narrative context, why the partnership matters, investment scale")

In [0]:
# =================================================
# Layer 3 only: Metadata — precise but narrow
# =================================================
metadata_results = spark.sql(f"""
    SELECT company, ai_investment_category, SUM(document_count) AS total_docs
    FROM {shared_schema}.company_ai_investment_summary
    WHERE ai_investment_category IS NOT NULL
    GROUP BY company, ai_investment_category
    ORDER BY company, total_docs DESC
""").collect()

print("Metadata layer alone — AI investment distribution:")
print(f"  company                   ai_investment_category          total_docs")
print(f"  -----------               ------------------------------  ----------")
for row in metadata_results:
    print(f"  {(row['company'] or ''):<25} {(row['ai_investment_category'] or ''):<30} {row['total_docs']:>3} docs")

print("\n⚠️  Missing: who the partners are, what the investments entail")

## Part 2: Build the hybrid context query function

Now let's combine all three layers into a single function that assembles the richest
possible context for answering a financial intelligence question.

In [0]:
import re

COMPANIES = ['Apple Inc.', 'Amazon.com, Inc.', 'Microsoft Corporation', 'NVIDIA Corporation', 'Meta Platforms, Inc.']

def hybrid_context_query(question, companies=None, max_chunks=10):
    """Assemble context from all three layers for a financial intelligence question.

    1. Vector layer:   Retrieve semantically relevant document chunks
    2. Entity extraction: Identify company names in retrieved chunks
    3. Graph layer:    Look up key relationships for identified entities
    4. Metadata layer: Add quantitative AI investment signals

    Returns a structured context string ready to pass to an LLM.
    """
    context_parts = []

    # ── Layer 1: Vector Search ────────────────────────────────────────────────
    company_filter = None
    if companies and len(companies) == 1:
        company_filter = f'{{"company": "{companies[0]}"}}'  # Filter if single company

    vs_results = w.vector_search_indexes.query_index(
        index_name=VS_INDEX_NAME,
        query_text=question,
        columns=["company", "fiscal_period", "document_type", "chunk_text"],
        filters_json=company_filter,
        num_results=max_chunks,
        query_type="HYBRID",
    )

    context_parts.append("## 📃 Relevant Document Excerpts (from Vector Search) 📃")
    retrieved_companies = set()
    for row in vs_results.result.data_array or []:
        company, period, doc_type, chunk = row[0], row[1], row[2], row[3]
        if company:
            retrieved_companies.add(company)
        context_parts.append(f"⬇️ [{company} | {period} | {doc_type}] ⬇️")
        context_parts.append(str(chunk))
        context_parts.append("⬆️  ⬆️\n")

    # ── Layer 2: Graph Traversal ──────────────────────────────────────────────
    print(f"Retrieved companies: {retrieved_companies}")
    target_companies = companies or list(retrieved_companies) or COMPANIES
    company_list = ", ".join(f"'{c}'" for c in target_companies)

    graph_data = spark.sql(f"""
        SELECT subject, predicate, object, COUNT(*) AS mentions
        FROM relationships
        WHERE subject IN ({company_list})
          AND predicate IN ('partners_with', 'invests_in', 'uses_technology')
        GROUP BY subject, predicate, object
        ORDER BY subject, mentions DESC
    """).collect()

    context_parts.append("\n\n## 🏭 Key Entity Relationships (from Knowledge Graph) 🏭")
    for row in graph_data[:20]:
        context_parts.append(f"{(row['subject'] or ''):<30} → {(row['predicate'] or ''):<20} → {(row['object'] or ''):<30} ({(row['mentions'] or '')} mentions)")

    # ── Layer 3: Metadata ─────────────────────────────────────────────────────
    metadata = spark.sql(f"""
        SELECT company, ai_investment_category, SUM(document_count) AS total_docs
        FROM {shared_schema}.company_ai_investment_summary
        WHERE company IN ({company_list})
          AND ai_investment_category != 'Not AI-related'
        GROUP BY company, ai_investment_category
        ORDER BY company, total_docs DESC
    """).collect()

    context_parts.append("\n\n## 💰 AI Investment Signals (from Metadata Layer) 💰")
    for row in metadata:
        context_parts.append(f"{row['company']:<30}: {row['ai_investment_category']:<30} — {row['total_docs']} documents")

    return context_parts


print("✓ hybrid_context_query() defined")

## Part 3: Answer the complex question

Now use the hybrid context to answer the question that stumped each individual layer.

In [0]:
# Retrieve the full context
context_parts = hybrid_context_query(QUESTION, max_chunks=25)

# Assemble the hybrid context
context = "".join([str(part)[:1000] for part in context_parts])

print(f"Context assembled: {len(context):,} characters from 3 layers")
print("\nContext preview (truncating chunks):")
# Print all context_parts, truncating to 500 characters for display
for i, part in enumerate(context_parts):
    print(str(part)[:500])

In [0]:
# Now send the assembled context + question to the Language Model
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

CHAT_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # Foundation Model API

prompt = f"""You are a financial analyst assistant. Use the following context to answer the question.
Be specific, cite companies by name, and highlight patterns across companies.

<context>
{context}
</context>

Question: {QUESTION}
"""

response = w.serving_endpoints.query(
    name=CHAT_ENDPOINT,
    messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
    max_tokens=1000,
)

print("=" * 60)
print("HYBRID CONTEXT RESPONSE")
print("=" * 60)
print(response.choices[0].message.content)

**Compare this response to what you got from the individual layers in Part 2.**

The hybrid response should:
- Name specific technology partners (from the graph layer)
- Quantify investment intensity (from the metadata layer)
- Provide narrative context and quotes (from the vector layer)
- Connect cross-company patterns that no single layer could reveal

## Key Takeaways

1. **No single layer is sufficient** for complex analytical questions — each has complementary strengths and weaknesses
2. **Vector search** provides semantic context; **graph** provides explicit relationships; **metadata** provides precise quantification
3. **Entity resolution** (finding company names in vector chunks, then looking them up in the graph) is the key integration mechanism
4. **Hybrid context assembly** is a design pattern, not just a retrieval strategy — it requires intentional data modelling upstream
5. **Databricks provides all three layers natively** — no third-party graph database, no separate vector store infrastructure

---

## Session 2 Complete

You've built the complete context layer for an AI-powered financial intelligence platform:
- ✓ **Vector layer** with Agent Bricks Knowledge Assistant
- ✓ **Graph layer** with entity-relationship triplets in Delta and Genie exploration
- ✓ **Metadata layer** from the AI-classified gold tables
- ✓ **Hybrid context** combining all three for richer, more accurate AI responses

---

## Next steps for a production deployment

1. **Productionise triplet extraction** — add it as a fourth Spark Declarative Pipeline table in the bundle
2. **Deploy as a Databricks App** — wrap `hybrid_context_query()` in a FastAPI or Streamlit app
3. **Add MLflow evaluation** — use `mlflow.genai.evaluate()` to measure retrieval quality and answer correctness
4. **Schedule freshness** — the pipeline + triplet extraction runs weekly after earnings release dates
5. **Extend to clients** — the same pattern applies to contract analysis, competitive intelligence, ESG reporting